In [17]:
from langgraph.graph import StateGraph,END,START
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from typing import TypedDict
import os
from langchain_groq import ChatGroq

In [18]:
load_dotenv(dotenv_path=".env", override=True)

True

In [19]:
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [20]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",  
    api_key=GROQ_API_KEY  
)

In [21]:
class jokestate(TypedDict):
    title:str
    jokes:str
    explaination:str


In [22]:
graph=StateGraph(jokestate)

In [23]:
def joke(state:jokestate)->jokestate:
    prompt=f"Generate the joke of the title{state['title']} in a funny way."
    response=model.invoke(prompt).content
    return {'jokes':response}

In [24]:
def explaination(state:jokestate)->jokestate:
    prompt=f"Explain the joke {state['jokes']}of the title{state['title']} in a funny way."
    response=model.invoke(prompt).content
    return {'explaination':response}

In [25]:
graph.add_node('joke',joke)
graph.add_node('explaination',explaination)

In [26]:
graph.add_edge(START,'joke')
graph.add_edge('joke','explaination')
graph.add_edge('explaination',END)
checkpoint=InMemorySaver()
workflow=graph.compile(checkpointer=checkpoint)

In [27]:
config1={'configurable':{'thread_id':'1'}}
workflow.invoke({'title':'samosa'},config=config1)

{'title': 'samosa',
 'jokes': 'Here\'s one:\n\nWhy did the samosa go to therapy?\n\nBecause it was feeling a little "crunchy" on the outside and "messy" on the inside, and it needed to work through some "filling" issues! (get it?)',
 'explaination': 'A clever joke. Let\'s break it down:\n\nThe joke is a play on words, using the characteristics of a samosa (a type of fried or baked pastry) to create a humorous connection to therapy.\n\n* "Crunchy on the outside" refers to the crispy exterior of a samosa, but also implies that the samosa is putting on a tough exterior, hiding its true feelings.\n* "Messy on the inside" describes the filling of a samosa, which can be loose and chaotic, but also suggests that the samosa is emotionally unstable or struggling with internal issues.\n* "Filling issues" is a double entendre, referencing both the filling of a samosa and the idea that the samosa has emotional or psychological issues to work through in therapy.\n\nThe joke relies on a lighthearted

In [28]:
workflow.get_state(config1)

StateSnapshot(values={'title': 'samosa', 'jokes': 'Here\'s one:\n\nWhy did the samosa go to therapy?\n\nBecause it was feeling a little "crunchy" on the outside and "messy" on the inside, and it needed to work through some "filling" issues! (get it?)', 'explaination': 'A clever joke. Let\'s break it down:\n\nThe joke is a play on words, using the characteristics of a samosa (a type of fried or baked pastry) to create a humorous connection to therapy.\n\n* "Crunchy on the outside" refers to the crispy exterior of a samosa, but also implies that the samosa is putting on a tough exterior, hiding its true feelings.\n* "Messy on the inside" describes the filling of a samosa, which can be loose and chaotic, but also suggests that the samosa is emotionally unstable or struggling with internal issues.\n* "Filling issues" is a double entendre, referencing both the filling of a samosa and the idea that the samosa has emotional or psychological issues to work through in therapy.\n\nThe joke relie